# Automazione dell'Analisi Documentale con LLM </p> Contesto Aziendale
L'azienda DataTrust Solutions, leader nel settore della consulenza finanziaria e compliance, gestisce quotidianamente un elevato volume di documenti regolatori, report finanziari e verbali di meeting aziendali. 
L'analisi manuale di questi documenti è onerosa e soggetta a errori umani, portando a inefficienze nei processi decisionali.

## Obiettivo del Progetto
Implementare un sistema basato su Large Language Models (LLMs) per automatizzare l'analisi dei documenti aziendali, estrarre insight chiave e generare riassunti strutturati. 
Il sistema sarà progettato per migliorare l'efficienza operativa, ridurre il rischio di non conformità e fornire informazioni utili ai decisori aziendali in tempo reale.

## Soluzione Tecnologica
Il progetto utilizza LangChain e LlamaIndex per interfacciarsi con i documenti aziendali, garantendo: 
- Ingestion intelligente: caricamento di documenti in vari formati (PDF, Word, e-mail, etc.)
- Elaborazione avanzata: estrazione automatica di dati chiave con tecniche di NLP
- Riassunto e categorizzazione: generazione di insight per area tematica (compliance, performance finanziaria, strategie aziendali, etc.).

## Vantaggi per l'Azienda
🚀 Efficienza Operativa
- Riduzione del tempo di analisi documentale
- Automazione della categorizzazione e della sintesi dei documenti

✅ Migliore Conformità Normativa
- Monitoraggio continuo dei requisiti di compliance
- Segnalazione automatica di eventuali anomalie o criticità nei documenti

📊 Decisioni più Veloci e Informate
- Disponibilità immediata di insight chiave senza necessità di analisi manuale
- Reportistica automatizzata per il management

## Implementazione
Il progetto si articola in 3 fasi: 
1. Setup e Integrazione: connessione del sistema ai repository aziendali e definizione delle pipeline di elaborazione
2. Sviluppo e Ottimizzazione: miglioramento della precisione degli LLM nel riconoscere pattern nei documenti
3. Validazione e Deployment: test su dataset reali e integrazione nel workflow aziendale.

Per questo progetto è possibile utilzzare dei documenti fittizi, che vanno rilasciati insieme al codice, estraendo alcuni insight chiave da essi. 
È importante documentare le varie fasi del progetto, la tipologia di documenti utilizzati e l'importanza strategica degli insight ricavati.

## Conclusioni
L'implementazione di un LLM per l'analisi documentale offre un vantaggio competitivo significativo a DataTrust Solutions, trasformando dati non strutturati in valore concreto per il business. 
Grazie a questa soluzione, l'azienda ottimizza i processi interni, garantisce la compliance e supporta il processo decisionale con informazioni accurate e tempestive.

## Setup e Integrazione

Gli embeddings e il vector store girano **in locale**, mentre l'LLM usa **ChatGroq** :

Componenti:
1. **LLM**: `llama3-70b-8192` via ChatGroq
2. **Embeddings**: `all-MiniLM-L6-v2`
3. **Vector Store**: ChromaDB
4. **Sparse Retrieval**: BM25

> **Prerequisito:** Creare un file `.env` nella directory del progetto con la propria API key Groq:
>
> ```
> GROQ_API_KEY=gsk_...
> ```

In [1]:
import os
import re
from pathlib import Path
from typing import List, Dict, Any, Optional, Tuple
from dataclasses import dataclass
from dotenv import load_dotenv

from IPython.display import display, Markdown

from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
import chromadb
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader, TextLoader


In [26]:
# Carica la API key dal file .env
load_dotenv()

if not os.getenv("GROQ_API_KEY"):
    raise EnvironmentError(
        "GROQ_API_KEY non trovata. Creare un file .env con: GROQ_API_KEY=gsk_..."
    )

# LLM via Groq
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
# Embedding model locale
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print(f"✅ LLM Groq: {llm.model_name}")
print(f"✅ Embedding model: all-MiniLM-L6-v2 (dim: {embedding_model.get_embedding_dimension()})")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ LLM Groq: llama-3.3-70b-versatile
✅ Embedding model: all-MiniLM-L6-v2 (dim: 384)


## Classi e Strutture Dati

- Chunker: `SearchResult` come contenitore dei risultati e `RecursiveChunker` per dividere i documenti in chunk indicizzabili.
- Caricamento Multi-Formato: `DocumentLoader` astrae la lettura di file in formati diversi usando i **LangChain Document Loaders**. Restituisce testi e metadati pronti per essere passati a `LocalRAGSystem.add_documents()`. È possibile caricare un singolo file oppure un'intera cartella.

In [3]:
@dataclass
class SearchResult:
    """Risultato di una ricerca nel vector store"""
    document_id: str  # ID univoco del chunk (es. "report_finanziario_chunk0")
    content: str  # Testo del chunk trovato
    score: float  # Score ibrido finale (0.0 → 1.0)
    metadata: Dict[str, Any]  # filename, chunk_index, ecc.


class RecursiveChunker:
    """
    Chunker: Divide il testo in chunk rispettando i separatori naturali:
    paragrafo → riga → frase → parola → carattere.
    Riceve testo già estratto da DocumentLoader (indipendente dal formato sorgente).
    """

    def __init__(self, chunk_size: int = 800, chunk_overlap: int = 100):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=["\n\n", "\n", ". ", " ", ""]
        )

    def split_text(self, text: str) -> List[str]:
        return self.splitter.split_text(text)


In [4]:
class DocumentLoader:
    """
    Carica documenti in vari formati (.txt, .pdf, .docx) usando i LangChain Document Loaders.
    Restituisce (texts, metadatas) pronti per LocalRAGSystem.add_documents().
    """

    # Mappa estensione -> classe loader LangChain
    _LOADERS = {
        ".txt": TextLoader,
        ".pdf": PyPDFLoader,
        ".docx": Docx2txtLoader,
        ".doc": Docx2txtLoader,
    }

    def __new__(cls, *args, **kwargs):
        raise TypeError("DocumentLoader e' una utility class statica e non puo' essere istanziata.")

    @staticmethod
    def load_file(filepath: str | Path) -> Tuple[List[str], List[Dict]]:
        """
        Carica un singolo file e restituisce (texts, metadatas).
        Per i PDF multi-pagina ogni pagina diventa un elemento separato.
        """
        path = Path(filepath)
        ext = path.suffix.lower()

        if ext not in DocumentLoader._LOADERS:
            raise ValueError(
                f"Formato '{ext}' non supportato. "
                f"Formati accettati: {list(DocumentLoader._LOADERS.keys())}"
            )

        loader_cls = DocumentLoader._LOADERS[ext]

        # TextLoader richiede encoding esplicito
        if ext == ".txt":
            loader = loader_cls(str(path), encoding="utf-8")
        else:
            loader = loader_cls(str(path))

        docs = loader.load()

        texts = [doc.page_content for doc in docs if doc.page_content.strip()]
        metadatas = [
            {
                "filename": path.name,
                "filepath": str(path),
                "format": ext.lstrip("."),
                "page": doc.metadata.get("page", 0),
            }
            for doc in docs if doc.page_content.strip()
        ]

        return texts, metadatas

    @staticmethod
    def load_directory(directory: str | Path,
                       extensions: Optional[List[str]] = None) -> Tuple[List[str], List[Dict]]:
        """
        Carica tutti i file supportati in una cartella.
        Se extensions e' specificato (es. ['.pdf', '.txt']), carica solo quei formati.
        """
        dir_path = Path(directory)
        if not dir_path.is_dir():
            raise NotADirectoryError(f"'{directory}' non e' una cartella valida.")

        allowed_ext = extensions or list(DocumentLoader._LOADERS.keys())

        all_texts: List[str] = []
        all_metadatas: List[Dict] = []

        found = sorted(
            f for f in dir_path.iterdir()
            if f.is_file() and f.suffix.lower() in allowed_ext
        )

        if not found:
            print(f"Nessun file supportato trovato in '{directory}'")
            return [], []

        for filepath in found:
            try:
                texts, metadatas = DocumentLoader.load_file(filepath)
                all_texts.extend(texts)
                all_metadatas.extend(metadatas)
            except Exception as e:
                print(f"Errore caricando {filepath.name}: {e}")

        return all_texts, all_metadatas


### LocalRAGSystem

Sistema RAG ibrido con pipeline completa **Retrieval → Augmentation → Generation**:
- **Dense retrieval**: ChromaDB + SentenceTransformer
- **Sparse retrieval**: BM25 per la corrispondenza lessicale esatta
- **LLM**: ChatGroq per generare risposte in linguaggio naturale a partire dai chunk recuperati

Metodi:
- `add_documents`: Indicizza testi in ChromaDB (dense) + BM25 (sparse)
- `search`: Retrieval ibrido: 70% semantico + 30% lessicale. Restituisce i chunk più rilevanti con score
- `query`: Pipeline RAG completa. Chiama la funzione `search`, costruisce il contesto e genera una risposta testuale con l'LLM

### Prompt di Query RAG
`query_prompt` guida il modello nella fase di **generation** del ciclo RAG: riceve in input il `{context}` (i chunk recuperati da `search()`) e la `{question}` dell'utente, e produce una risposta ancorata esclusivamente al contesto fornito.

In [5]:
query_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """Sei un assistente esperto di DataTrust Solutions.
Rispondi alla domanda dell'utente basandoti ESCLUSIVAMENTE sui seguenti frammenti di documenti aziendali.
Se la risposta non è presente nel contesto, dì esplicitamente che non hai informazioni sufficienti.

CONTESTO:
{context}""",
    ),
    (
        "human",
        "{question}",
    ),
])


In [6]:
class LocalRAGSystem:
    """
    Sistema RAG ibrido per l'analisi documentale aziendale.
    Pipeline: add_documents → search → query
    """

    def __init__(self, collection_name: str = "datatrust_docs"):
        self.embedding_model = embedding_model
        self.chroma_client = chromadb.Client()
        try:
            self.chroma_client.delete_collection(collection_name)
        except Exception:
            pass
        self.collection = self.chroma_client.create_collection(
            name=collection_name,
            metadata={"hnsw:space": "cosine"}
        )
        self.bm25: Optional[BM25Okapi] = None
        self.chunks: List[str] = []
        self.chunk_ids: List[str] = []
        self.chunker = RecursiveChunker(chunk_size=800, chunk_overlap=100)
        self.llm = llm

    # ------------------------------------------------------------------
    # Indicizzazione
    # ------------------------------------------------------------------

    def add_documents(self, texts: List[str], metadatas: Optional[List[Dict]] = None):
        """
        Carica una lista di testi nel sistema:
        1. Chunking ricorsivo
        2. Encoding in embedding
        3. Salvataggio in ChromaDB
        4. Costruzione indice BM25
        """
        if metadatas is None:
            metadatas = [{"doc_id": i, "filename": f"doc_{i}"} for i in range(len(texts))]

        all_chunks, all_metadatas = [], []
        for i, text in enumerate(texts):
            for j, chunk in enumerate(self.chunker.split_text(text)):
                all_chunks.append(chunk)
                all_metadatas.append({
                    **metadatas[i],
                    "chunk_id": f"{metadatas[i].get('filename', f'doc{i}')}_chunk{j}",
                    "chunk_index": j,
                })

        # Dense: encoding + salvataggio ChromaDB
        embeddings = self.embedding_model.encode(all_chunks, show_progress_bar=True)
        ids = [m["chunk_id"] for m in all_metadatas]
        self.collection.add(embeddings=embeddings.tolist(), documents=all_chunks,
                            metadatas=all_metadatas, ids=ids)

        # Sparse: costruzione indice BM25
        self.chunks = all_chunks
        self.chunk_ids = ids
        self.bm25 = BM25Okapi([self._tokenize(c) for c in all_chunks])

    @staticmethod
    def _tokenize(text: str) -> List[str]:
        return [t for t in re.findall(r'\b\w+\b', text.lower()) if len(t) > 2]

    def search(self, query: str, k: int = 3, hybrid_weight: float = 0.7) -> List[SearchResult]:
        """
        Ricerca ibrida: combina score dense (ChromaDB) e sparse (BM25).
        hybrid_weight = 0.7 → 70% semantico, 30% lessicale.
        """
        # Dense search
        q_emb = self.embedding_model.encode([query])[0]
        n = min(k * 2, len(self.chunks)) if self.chunks else k
        dense_res = self.collection.query(query_embeddings=[q_emb.tolist()], n_results=n)

        # Sparse search
        bm25_scores = self.bm25.get_scores(self._tokenize(query)) if self.bm25 else []
        max_bm25 = max(bm25_scores) if len(bm25_scores) > 0 else 1.0

        # Mappa chunk_id -> BM25
        bm25_map = {self.chunk_ids[i]: float(s) for i, s in enumerate(bm25_scores)}

        # Fusione
        merged: Dict[str, Dict] = {}
        if dense_res["ids"] and dense_res["ids"][0]:
            for i, cid in enumerate(dense_res["ids"][0]):
                merged[cid] = {
                    "content": dense_res["documents"][0][i],
                    "metadata": dense_res["metadatas"][0][i],
                    "dense_score": 1 - dense_res["distances"][0][i],
                    "sparse_score": bm25_map.get(cid, 0.0),
                }

        final = []
        for cid, data in merged.items():
            score = (hybrid_weight * data["dense_score"]
                     + (1 - hybrid_weight) * data["sparse_score"] / (max_bm25 + 1e-8))
            final.append(SearchResult(document_id=cid, content=data["content"],
                                      score=score, metadata=data["metadata"]))
        final.sort(key=lambda x: x.score, reverse=True)
        return final[:k]

    def query(self, question: str, k: int = 3) -> str:
        """
        Pipeline RAG completa:
        1. Retrieval: search() recupera i k chunk più rilevanti alla domanda
        2. Augmentation: i chunk vengono concatenati come contesto nel prompt
        3. Generation: l'LLM risponde basandosi SOLO sul contesto recuperato
        """
        results = self.search(question, k=k)
        if not results:
            return "Nessun documento rilevante trovato per questa domanda."

        context = "\n\n---\n\n".join(
            f"[Fonte: {r.metadata.get('filename', 'documento')} | score: {r.score:.3f}]\n{r.content}"
            for r in results
        )
        return (query_prompt | self.llm | StrOutputParser()).invoke(
            {"context": context, "question": question}
        )


## Caricamento Documenti e Indicizzazione

Per simulare il contesto operativo di DataTrust Solutions vengono creati **tre documenti fittizi** in formato `.txt`, ciascuno rappresentativo di una diversa area documentale aziendale:

| File | Categoria |
|---|---|
| `report_finanziario_Q3_2024.txt` | Finanziario |
| `verbale_meeting_strategico.txt` | Strategico |
| `assessment_compliance_GDPR.txt` | Compliance |

Una volta scritti nella cartella `documents/`, il flusso di indicizzazione procede in due passi:
1. `DocumentLoader.load_directory()` legge tutti i file supportati dalla cartella e restituisce testi e metadati normalizzati
2. `rag.add_documents()` applica il chunking ricorsivo, genera gli embedding locali e indicizza i chunk in **ChromaDB** e **BM25**


In [7]:
docs_dir = Path("documents")
docs_dir.mkdir(exist_ok=True)

DOCUMENTS = {
    "report_finanziario_Q3_2024.txt": """\
DATATRUST SOLUTIONS — REPORT FINANZIARIO Q3 2024
Periodo: 1 Luglio – 30 Settembre 2024

EXECUTIVE SUMMARY
Il terzo trimestre 2024 conferma il trend positivo di DataTrust Solutions con ricavi totali di 8,4 milioni di euro,
in crescita del 12% rispetto al Q3 2023. Il margine EBITDA si attesta al 23%, in linea con le previsioni annuali.

RISULTATI FINANZIARI CONSOLIDATI
- Ricavi totali: EUR 8,4 milioni (+12% YoY)
- EBITDA: EUR 1,93 milioni (margine: 23%)
- Utile netto: EUR 1,1 milioni (+8% YoY)
- Cash flow operativo: EUR 2,1 milioni
- Posizione finanziaria netta: EUR 4,5 milioni (+EUR 0,8M vs Q2 2024)

DETTAGLIO PER AREA DI BUSINESS
1. Consulenza Compliance: EUR 3,2 milioni (+18% YoY) — trainata da DORA e AI Act
2. Analisi del Rischio: EUR 2,8 milioni (+10% YoY) — nuovi mandati bancari e assicurativi
3. Servizi IT e Data Analytics: EUR 2,4 milioni (+7% YoY) — 4 nuovi contratti SaaS

COSTI OPERATIVI
- Costo del personale: EUR 4,2 milioni (+9% YoY)
- Costi tecnologici: EUR 0,9 milioni (+15% YoY, investimenti AI)
- Costi generali e amministrativi: EUR 1,3 milioni (stabili)

OUTLOOK Q4 2024
Si prevede una crescita dei ricavi del 15% nel Q4, sostenuta da:
- 3 nuovi clienti enterprise nel settore bancario (EUR 2,1M annui)
- Pipeline commerciale attiva: EUR 15 milioni
- Rinnovi contrattuali attesi: EUR 5,2 milioni

RISCHI IDENTIFICATI
1. Intensificazione della concorrenza con 5 nuovi player nel compliance consulting
2. Possibile rallentamento degli investimenti IT nel contesto macro attuale
3. Rischio cambio: EUR 0,3M di ricavi in GBP

OPPORTUNITA' STRATEGICHE
1. Regolamento DORA dal 17 gennaio 2025: prevista impennata della domanda
2. AI Act europeo: mercato stimato EUR 50M nel triennio 2025-2027
3. Consolidamento del mercato: possibile acquisizione di un competitor
""",

    "verbale_meeting_strategico.txt": """\
DATATRUST SOLUTIONS — VERBALE MEETING STRATEGICO
Data: 15 Ottobre 2024 | Ore 09:00–13:00 | Sede: Milano

PARTECIPANTI
- Marco Rossi — CEO
- Elena Bianchi — CFO
- Alessandro Ferrari — CTO
- Francesca Marino — Chief Compliance Officer (CCO)
- Luca Conti — HR Director
- Sofia Pellegrini — Chief Strategy Officer

DELIBERE

1. REVIEW PERFORMANCE Q3 2024
Approvazione risultati Q3 e conferma budget Q4.
La CFO ha evidenziato la necessita' di ottimizzare i costi IT nel Q4 per preservare il margine al 23%.

2. PIANO STRATEGICO 2025-2027
a) CRESCITA GEOGRAFICA: apertura sedi a Francoforte e Madrid entro Q2 2025 (EUR 2M).
   Target: 20% dei ricavi da mercati esteri entro fine 2026.
b) INNOVAZIONE AI: sviluppo piattaforma LLM proprietaria (budget R&D EUR 1,5M nel 2025).
   Go-to-market previsto Q3 2025.
c) CRESCITA INORGANICA: acquisizione software house RegTech (valutazione EUR 5-8M).

3. COMPLIANCE E REGOLAMENTAZIONE
Stato di conformita' GDPR all'87%. Identificati 3 gap critici da sanare entro Q1 2025.
Investimento stimato: EUR 350.000. Priorità ALTA per chiusura gap DORA.

4. PIANO ASSUNZIONI 2025
45 nuove risorse: data scientist (10), compliance specialist (12), cloud engineer (8),
business developer (8), staff amministrativo (7).

ACTION ITEMS
- Marco Rossi: Finalizzare term sheet acquisizione RegTech — entro 30/11/2024
- Elena Bianchi: Presentare budget 2025 al CDA — entro 15/11/2024
- Alessandro Ferrari: Avviare due diligence tecnologica — entro 30/11/2024
- Francesca Marino: Piano di remediation DORA — entro 01/11/2024
- Luca Conti: Pubblicare 15 posizioni prioritarie — entro 25/10/2024
""",

    "assessment_compliance_GDPR.txt": """\
DATATRUST SOLUTIONS — GDPR COMPLIANCE ASSESSMENT 2024
Documento: DTS-GDPR-2024-001 | Classificazione: RISERVATO
DPO: Dott.ssa Giulia Moretti (CIPP/E)

EXECUTIVE SUMMARY
Livello di conformita' complessivo: 87% (vs 82% del 2023).
Identificati 4 gap da sanare entro H1 2025. Investimento totale: EUR 310.000.

STATO DI CONFORMITA' PER AREA

2.1 BASI GIURIDICHE E CONSENSI — CONFORME (95%)
OK: Basi giuridiche per tutti i 52 trattamenti censiti nel Registro Art. 30 GDPR
OK: Sistema di gestione consensi implementato su tutti i canali digitali

2.2 DATA SECURITY — PARZIALMENTE CONFORME (80%)
OK: Cifratura AES-256 per dati at-rest su tutti i sistemi
OK: 3 penetration test nel 2024, nessuna vulnerabilita' critica
GAP-1 (ALTA PRIORITÀ): Cifratura TLS 1.3 non completata su 3 sistemi legacy

2.3 GESTIONE DATA BREACH — CONFORME (92%)
OK: Tempo medio di notifica al Garante: 48 ore (limite GDPR: 72 ore)
OK: Zero data breach con obbligo di notifica nel 2024

2.4 DIRITTI DEGLI INTERESSATI — PARZIALMENTE CONFORME (78%)
OK: 15 richieste DSAR gestite nel 2024, tempo medio risposta 25 giorni
GAP-2 (MEDIA PRIORITÀ): Processo Right to Erasure ancora manuale
GAP-3 (BASSA PRIORITÀ): Assenza di portale self-service per gli interessati

2.5 FORMAZIONE — PARZIALMENTE CONFORME (75%)
GAP-4 (MEDIA PRIORITÀ): Formazione GDPR non aggiornata per 45 risorse tecniche

GAP ANALYSIS E PIANO DI REMEDIATION
GAP-1 | TLS 1.3 sistemi legacy    | Impatto: ALTO  | Deadline: 31/12/2024 | EUR 80.000
GAP-2 | Automazione Right to Erasure | Impatto: MEDIO | Deadline: 31/03/2025 | EUR 120.000
GAP-3 | Portale self-service       | Impatto: BASSO | Deadline: 30/06/2025 | EUR 95.000
GAP-4 | Formazione personale tecnico | Impatto: MEDIO | Deadline: 28/02/2025 | EUR 15.000

RACCOMANDAZIONI
1. Prioritizzare GAP-1 per evitare sanzioni GDPR (fino al 4% del fatturato)
2. Adottare piattaforma Privacy Management per automatizzare DSAR
3. Integrare requisiti GDPR nella nuova piattaforma AI (Privacy by Design)
""",
}

for filename, content in DOCUMENTS.items():
    (docs_dir / filename).write_text(content, encoding="utf-8")
    print(f"Documento creato: documents/{filename}")

print(f"\n{len(DOCUMENTS)} documenti salvati in 'documents/'")


Documento creato: documents/report_finanziario_Q3_2024.txt
Documento creato: documents/verbale_meeting_strategico.txt
Documento creato: documents/assessment_compliance_GDPR.txt

3 documenti salvati in 'documents/'


In [8]:
texts, metadatas = DocumentLoader.load_directory("documents/")

rag = LocalRAGSystem()
rag.add_documents(texts, metadatas)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

## Retrieval - Validazione di `search()`

`search()` è il motore del sistema RAG: data una query in linguaggio naturale, recupera i chunk più rilevanti tramite retrieval **ibrido** (70% semantico + 30% lessicale BM25).

Questa cella mostra i chunk restituiti con il relativo score ibrido, permettendo di verificare che l'indicizzazione abbia funzionato correttamente e che i frammenti più pertinenti vengano effettivamente recuperati.


In [25]:
query = "Quali sono i gap di conformità GDPR da sanare con urgenza?"
results = rag.search(query, k=3)

display(Markdown(f"**Query**: {query}\n\n **Top {len(results)} chunk recuperati:**"))
for i, r in enumerate(results, 1):
    display(Markdown(
        f"**#{i}** | `{r.document_id}` | Score ibrido: `{r.score:.3f}`\n\n"
    ))
    display(Markdown(f">{r.content.replace('\n', '\n> ')}"))

    display(Markdown("---"))



**Query**: Quali sono i gap di conformità GDPR da sanare con urgenza?

 **Top 3 chunk recuperati:**

**#1** | `assessment_compliance_GDPR.txt_chunk0` | Score ibrido: `0.665`



>DATATRUST SOLUTIONS — GDPR COMPLIANCE ASSESSMENT 2024
> Documento: DTS-GDPR-2024-001 | Classificazione: RISERVATO
> DPO: Dott.ssa Giulia Moretti (CIPP/E)
> 
> EXECUTIVE SUMMARY
> Livello di conformita' complessivo: 87% (vs 82% del 2023).
> Identificati 4 gap da sanare entro H1 2025. Investimento totale: EUR 310.000.
> 
> STATO DI CONFORMITA' PER AREA
> 
> 2.1 BASI GIURIDICHE E CONSENSI — CONFORME (95%)
> OK: Basi giuridiche per tutti i 52 trattamenti censiti nel Registro Art. 30 GDPR
> OK: Sistema di gestione consensi implementato su tutti i canali digitali
> 
> 2.2 DATA SECURITY — PARZIALMENTE CONFORME (80%)
> OK: Cifratura AES-256 per dati at-rest su tutti i sistemi
> OK: 3 penetration test nel 2024, nessuna vulnerabilita' critica
> GAP-1 (ALTA PRIORITÀ): Cifratura TLS 1.3 non completata su 3 sistemi legacy

---

**#2** | `verbale_meeting_strategico.txt_chunk1` | Score ibrido: `0.651`



>2. PIANO STRATEGICO 2025-2027
> a) CRESCITA GEOGRAFICA: apertura sedi a Francoforte e Madrid entro Q2 2025 (EUR 2M).
>    Target: 20% dei ricavi da mercati esteri entro fine 2026.
> b) INNOVAZIONE AI: sviluppo piattaforma LLM proprietaria (budget R&D EUR 1,5M nel 2025).
>    Go-to-market previsto Q3 2025.
> c) CRESCITA INORGANICA: acquisizione software house RegTech (valutazione EUR 5-8M).
> 
> 3. COMPLIANCE E REGOLAMENTAZIONE
> Stato di conformita' GDPR all'87%. Identificati 3 gap critici da sanare entro Q1 2025.
> Investimento stimato: EUR 350.000. Priorità ALTA per chiusura gap DORA.
> 
> 4. PIANO ASSUNZIONI 2025
> 45 nuove risorse: data scientist (10), compliance specialist (12), cloud engineer (8),
> business developer (8), staff amministrativo (7).

---

**#3** | `assessment_compliance_GDPR.txt_chunk1` | Score ibrido: `0.599`



>2.3 GESTIONE DATA BREACH — CONFORME (92%)
> OK: Tempo medio di notifica al Garante: 48 ore (limite GDPR: 72 ore)
> OK: Zero data breach con obbligo di notifica nel 2024
> 
> 2.4 DIRITTI DEGLI INTERESSATI — PARZIALMENTE CONFORME (78%)
> OK: 15 richieste DSAR gestite nel 2024, tempo medio risposta 25 giorni
> GAP-2 (MEDIA PRIORITÀ): Processo Right to Erasure ancora manuale
> GAP-3 (BASSA PRIORITÀ): Assenza di portale self-service per gli interessati
> 
> 2.5 FORMAZIONE — PARZIALMENTE CONFORME (75%)
> GAP-4 (MEDIA PRIORITÀ): Formazione GDPR non aggiornata per 45 risorse tecniche

---

## Generation - Pipeline RAG con `query()`

`query()` orchestra l'intero ciclo **Retrieval → Augmentation → Generation**:
1. Chiama `search()` per recuperare i chunk più pertinenti
2. Li concatena come contesto con indicazione della fonte e dello score
3. Invia contesto + domanda a ChatGroq tramite `query_prompt`


In [22]:
domande = [
    "Quali sono i ricavi totali del Q3 2024 e la crescita rispetto all'anno precedente?",
    "Quali gap GDPR richiedono intervento urgente e quanto costeranno?",
    "Quali sono le priorità del piano strategico 2025-2027?",
]

for domanda in domande:
    display(Markdown(f"---\n**{domanda}**"))
    risposta = rag.query(domanda, k=3)
    display(Markdown(f"**Risposta (RAG):** {risposta}"))

display(Markdown("---"))


---
**Quali sono i ricavi totali del Q3 2024 e la crescita rispetto all'anno precedente?**

**Risposta (RAG):** I ricavi totali del Q3 2024 sono di 8,4 milioni di euro, con una crescita del 12% rispetto al Q3 2023.

---
**Quali gap GDPR richiedono intervento urgente e quanto costeranno?**

**Risposta (RAG):** Secondo il documento di valutazione della conformità GDPR, ci sono 4 gap identificati, ma solo alcuni di essi richiedono un intervento urgente.

In particolare, il GAP-1 relativo alla cifratura TLS 1.3 sui sistemi legacy ha un impatto alto e una scadenza del 31/12/2024, quindi richiede un intervento urgente. Il costo stimato per la risoluzione di questo gap è di 80.000 euro.

Inoltre, il GAP-2 relativo all'automazione del diritto di cancellazione (Right to Erasure) ha un impatto medio e una scadenza del 31/03/2025, quindi potrebbe essere considerato come richiedente un intervento relativamente urgente. Il costo stimato per la risoluzione di questo gap è di 120.000 euro.

Il GAP-4 relativo alla formazione del personale tecnico ha un impatto medio e una scadenza del 28/02/2025, quindi potrebbe essere considerato come richiedente un intervento relativamente urgente. Il costo stimato per la risoluzione di questo gap è di 15.000 euro.

In sintesi, i gap che richiedono un intervento urgente sono:

* GAP-1: cifratura TLS 1.3 sui sistemi legacy (impatto alto, scadenza 31/12/2024, costo 80.000 euro)
* GAP-2: automazione del diritto di cancellazione (impatto medio, scadenza 31/03/2025, costo 120.000 euro)
* GAP-4: formazione del personale tecnico (impatto medio, scadenza 28/02/2025, costo 15.000 euro)

Il totale dei costi stimati per la risoluzione di questi gap è di 215.000 euro.

---
**Quali sono le priorità del piano strategico 2025-2027?**

**Risposta (RAG):** Le priorità del piano strategico 2025-2027 di DataTrust Solutions sono:

1. **Crescita geografica**: apertura di sedi a Francoforte e Madrid entro il Q2 2025, con l'obiettivo di raggiungere il 20% dei ricavi da mercati esteri entro la fine del 2026.
2. **Innovazione AI**: sviluppo di una piattaforma LLM proprietaria, con un budget R&D di 1,5 milioni di euro nel 2025 e un go-to-market previsto per il Q3 2025.
3. **Crescita inorganica**: acquisizione di una software house RegTech, con una valutazione di 5-8 milioni di euro.

Inoltre, è stata identificata una priorità alta per la chiusura dei gap critici di conformità GDPR e DORA, con un investimento stimato di 350.000 euro e un termine di scadenza per il Q1 2025.

---